In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully
Table VERSION_TRACKER created successfully
Table METRICS_TRACKER created successfully

Share anonymous install statistics? (opt-out instructions)

PixieDust will record metadata on its environment the next time the package is installed or updated. The data is anonymized and aggregated to help plan for future releases, and records only the following values:

{
   "data_sent": currentDate,
   "runtime": "python",
   "application_version": currentPixiedustVersion,
   "space_id": nonIdentifyingUniqueId,
   "config": {
       "repository_id": "https://github.com/ibm-watson-data-lab/pixiedust",
       "target_runtimes": ["Data Science Experience"],
       "event_id": "web",
       "event_organizer": "dev-journeys"
   }
}
You can opt out by calling pixiedust.optOut() in a new cell.


Pixiedust runtime updated. Please restart kernel
Table SPARK_PACKAGES created successfully
Table USER_PREFERENCES created successfully
Table service_connections created successfully
Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
     |████████████████████████████████| 20.2 MB 6.9 MB/s eta 0:00:01
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


Exception in thread Thread-6:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.7/threading.py", line 926, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.7/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 47, in startSparkJobProgressMonitor
    progressMonitor = SparkJobProgressMonitor()
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 174, in __init__
    self.addSparkListener()
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 203, in addSparkListener
    _env.getTemplate("sparkJobProgressMonitor/addSparkListener.scala").render()
  File "/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py", line 2352, in run_cell_magic
    result = fn(*args, **kwargs)
  File "</opt/conda/lib/python3.7/site-packages/decorator.py:d

In [1]:
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-superset-finalV1")

In [3]:
Epilepsy_Combined.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- MedicalHistory: double (nullable = true)
 |-- race_vector: vector (nullable = true)
 |-- gender_vector: vector (nullable = true)
 |-- E72: double (nullable = true)
 |-- Q02: double (nullable = true)
 |-- S37: double (nullable = true)
 |-- R52: double (nullable = true)
 |-- L74: double (nullable = true)
 |-- T50: double (nullable = true)
 |-- S87: double (nullable = true)
 |-- S41: double (nullable = true)
 |-- C25: double (nullable = true)
 |-- V58: double (nullable = true)
 |-- B37: double (nullable = true)
 |-- T58: double (nullable = true)
 |-- V41: double (nullable = true)
 |-- X34: double (nullable = true)
 |-- C86: double (nullable = true)
 |-- I01: double (nullable = true)
 |-- Q11: double (nullable = true)
 |-- M75: double (nullable = true)
 |-- B70: double (nullable = true)
 |-- Q52: double (nullable = true)
 |-- W27: double (nullable = true)
 |-- R36: double (nullable = true)

In [2]:
balanced_df = Epilepsy_Combined.drop("personid")

In [3]:
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions = balanced_df.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    balanced_df = balanced_df.repartition(200)
    print(f"DataFrame repartitioned into 200 partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")

DataFrame repartitioned into 200 partitions successfully.


In [9]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
import random
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, VectorSizeHint

# Step 0: Mimicking randomSplit with seed and randomness
def probabilistic_split(df: DataFrame, fractions: list, seed=None) -> list:
    if seed is not None:
        random.seed(seed)

    cumulative_fractions = [sum(fractions[:i + 1]) for i in range(len(fractions))]
    random_col = F.rand(seed)
    df_with_random = df.withColumn("random", random_col)
    splits = []
    prev_fraction = 0
    for fraction in cumulative_fractions:
        split_df = df_with_random.filter((F.col("random") >= prev_fraction) & (F.col("random") < fraction))
        splits.append(split_df.drop("random"))
        prev_fraction = fraction
    return splits

# Train, validation, and test sampling fractions
train_fraction = 0.7
valid_fraction = 0.2
test_fraction = 0.1
fractions = [train_fraction, valid_fraction, test_fraction]
seed_value = 23
train_data, valid_data, test_data = probabilistic_split(balanced_df, fractions, seed=seed_value)

# Show class distribution in train, validation, and test datasets
# train_data.groupBy('label').count().show()
# valid_data.groupBy('label').count().show()
# test_data.groupBy('label').count().show()

# # Print counts
# print("Sampled data count:", balanced_df.count())
# print("Train data count:", train_data.count())
# print("Validation data count:", valid_data.count())
# print("Test data count:", test_data.count())

In [4]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import random

def probabilistic_split(df: DataFrame, fractions: list, label_col="label", seed=None) -> list:
    """
    Stratified probabilistic split mimicking `randomSplit`, ensuring balanced class distribution.

    :param df: Input DataFrame
    :param fractions: List of fractions summing to 1 (e.g., [0.7, 0.2, 0.1])
    :param label_col: Column name representing class labels
    :param seed: Random seed for reproducibility
    :return: List of DataFrames split into train, validation, and test
    """
    if seed is not None:
        random.seed(seed)

    cumulative_fractions = [sum(fractions[:i + 1]) for i in range(len(fractions))]
    random_col = F.rand(seed)

    # Ensure each class is split proportionally
    splits = []
    
    for label in df.select(label_col).distinct().rdd.flatMap(lambda x: x).collect():
        df_label = df.filter(F.col(label_col) == label).withColumn("random", random_col)
        prev_fraction = 0
        class_splits = []
        
        for fraction in cumulative_fractions:
            split_df = df_label.filter((F.col("random") >= prev_fraction) & (F.col("random") < fraction))
            class_splits.append(split_df.drop("random"))
            prev_fraction = fraction
        
        if not splits:
            splits = class_splits
        else:
            splits = [s.union(class_splits[i]) for i, s in enumerate(splits)]

    return splits

# Train, validation, and test sampling fractions
fractions = [0.7, 0.2, 0.1]
seed_value = 23

# Stratified split
train_data, valid_data, test_data = probabilistic_split(balanced_df, fractions, label_col="label", seed=seed_value)

In [5]:
def check_class_distribution(df, label_col="label"):
    return df.groupBy(label_col).count().orderBy(label_col).show()

print("Train Data:")
check_class_distribution(train_data)
print("Validation Data:")
check_class_distribution(valid_data)
print("Test Data:")
check_class_distribution(test_data)

Train Data:
+-----+------+
|label| count|
+-----+------+
|  0.0|706360|
|  1.0|106714|
+-----+------+

Validation Data:
+-----+------+
|label| count|
+-----+------+
|  0.0|201779|
|  1.0| 30501|
+-----+------+

Test Data:
+-----+------+
|label| count|
+-----+------+
|  0.0|100724|
|  1.0| 15185|
+-----+------+



In [6]:
def check_class_proportions(df_list, split_names):
    """
    Prints the proportion of each class in different dataset splits.
    """
    for df, name in zip(df_list, split_names):
        total_count = df.count()
        class_counts = df.groupBy("label").count().withColumn("proportion", F.col("count") / total_count)
        print(f"\n{name} Split Class Proportions:")
        class_counts.show()

# Compare proportions
check_class_proportions([train_data, valid_data, test_data], ["Train", "Validation", "Test"])


Train Split Class Proportions:
+-----+------+------------------+
|label| count|        proportion|
+-----+------+------------------+
|  0.0|706360|0.8687524136794437|
|  1.0|106714|0.1312475863205563|
+-----+------+------------------+


Validation Split Class Proportions:
+-----+------+-------------------+
|label| count|         proportion|
+-----+------+-------------------+
|  0.0|201779| 0.8686886516273463|
|  1.0| 30501|0.13131134837265368|
+-----+------+-------------------+


Test Split Class Proportions:
+-----+------+-----------------+
|label| count|       proportion|
+-----+------+-----------------+
|  0.0|100724|0.868992054111415|
|  1.0| 15185|0.131007945888585|
+-----+------+-----------------+



In [9]:
from pyspark.ml.linalg import Vectors
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, ArrayType

# Function to check feature distribution, including vector columns and other feature types
def check_feature_distribution(df_list, split_names, features):
    for df, name in zip(df_list, split_names):
        print(f"\nFeature Statistics for {name} Split:")
        
        # Separate vector columns from numeric columns
        numeric_columns = [f for f in features if isinstance(df.schema[f].dataType, NumericType)]
        vector_columns = [f for f in features if isinstance(df.schema[f].dataType, ArrayType)]
        
        # Compute statistics for numeric columns
        numeric_stats = df.select(
            *[F.mean(F.col(f)).alias(f"{f}_mean") for f in numeric_columns] +
            [F.stddev(F.col(f)).alias(f"{f}_std") for f in numeric_columns]
        )
        
        print("Numeric Features Statistics:")
        numeric_stats.show(truncate=False)

        # Decompose and compute statistics for vector columns
        for vector_col in vector_columns:
            # Decompose vector column into individual features (elements)
            decomposed_df = df.withColumn(f"{vector_col}_components", F.explode(F.col(vector_col)))
            vector_stats = decomposed_df.select(
                F.mean(F.col(f"{vector_col}_components")).alias(f"{vector_col}_mean"),
                F.stddev(F.col(f"{vector_col}_components")).alias(f"{vector_col}_std")
            )
            
            print(f"Vector Column: {vector_col}")
            vector_stats.show(truncate=False)

# Select a subset of features (modify this based on your dataset)
selected_features = balanced_df.columns[:10]  # Choose the first 10 features or select relevant features

# Run the feature distribution check
check_feature_distribution([train_data, valid_data, test_data], ["Train", "Validation", "Test"], selected_features)


Feature Statistics for Train Split:
Numeric Features Statistics:
+-------------------------+-------------------+---------------------+--------------------+-------------------+-------------------+---------------------+--------------------+------------------------+------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+
|age_of_TBI_diagnosis_mean|MedicalHistory_mean|E72_mean             |Q02_mean            |S37_mean           |R52_mean           |L74_mean             |T50_mean            |age_of_TBI_diagnosis_std|MedicalHistory_std|E72_std            |Q02_std             |S37_std            |R52_std            |L74_std            |T50_std            |
+-------------------------+-------------------+---------------------+--------------------+-------------------+-------------------+---------------------+--------------------+------------------------+------------------+-------------------+-------------------

In [10]:
train_data.rdd.getNumPartitions()

400

In [11]:
train_data.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train2')

In [11]:
train_data.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train3')

In [12]:
valid_data.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-valid2')

In [12]:
valid_data.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-valid3')

In [13]:
test_data.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-test2')

In [13]:
test_data.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-test3')

In [1]:
# train_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train2')
train_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train3')

In [2]:
# valid_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-valid2')
# test_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-test2')
valid_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-valid3')
test_data = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-test3')

In [3]:
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [4]:
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and transform data
def apply_vector_size_hint(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 1% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")

        # Apply VectorSizeHint if the vector size is greater than 0
        if vector_size > 0:
            size_hint = VectorSizeHint(inputCol=col_name, size=vector_size)
            pipeline_size_hint = Pipeline(stages=[size_hint])
            
            print(f"Fitting VectorSizeHint for column '{col_name}' on sample of {sample_fraction * 100}% of data...")
            model_size_hint = pipeline_size_hint.fit(sampled_df)
            
            print(f"Applying VectorSizeHint for column '{col_name}' on full dataset...")
            return model_size_hint.transform(data)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")

    return data

# Applying the function to each vector column
for col_name in vector_cols:
    print(f"Applying VectorSizeHint to vector column: '{col_name}'")
    train_data = apply_vector_size_hint(train_data, col_name)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")
pipeline_final_assembler = Pipeline(stages=[final_assembler])
model_final_assembler = pipeline_final_assembler.fit(train_data)
print("Pipeline Processing Done")

# Step 6: Transform train, valid, and test datasets to include the final features
train_data = model_final_assembler.transform(train_data)

print("Pipeline Transformation Done")

Applying VectorSizeHint to vector column: 'race_vector'
Column 'race_vector' vector size: 1
Fitting VectorSizeHint for column 'race_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'race_vector' on full dataset...
Applying VectorSizeHint to vector column: 'gender_vector'
Column 'gender_vector' vector size: 1
Fitting VectorSizeHint for column 'gender_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'gender_vector' on full dataset...
Final input columns for feature assembly: ['race_vector', 'gender_vector', 'age_of_TBI_diagnosis', 'MedicalHistory', 'E72', 'Q02', 'S37', 'R52', 'L74', 'T50', 'S87', 'S41', 'C25', 'V58', 'B37', 'T58', 'V41', 'X34', 'C86', 'I01', 'Q11', 'M75', 'B70', 'Q52', 'W27', 'R36', 'V65', 'V28', 'I47', 'K87', 'I52', 'E86', 'Z33', 'Q23', 'Q86', 'N46', 'L62', 'V97', 'T30', 'L50', 'S56', 'H62', 'C91', 'G70', 'Q45', 'T27', 'Z77', 'S51', 'P05', 'F29', 'A83', 'F20', 'W89', 'B68', 'O72', 'X77', 'L87', 'X39', 'B42', 'R59', 'V38', 

Pipeline Transformation Done


In [9]:
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and transform data
def apply_vector_size_hint(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 1% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")

        # Apply VectorSizeHint if the vector size is greater than 0
        if vector_size > 0:
            size_hint = VectorSizeHint(inputCol=col_name, size=vector_size)
            pipeline_size_hint = Pipeline(stages=[size_hint])
            
            print(f"Fitting VectorSizeHint for column '{col_name}' on sample of {sample_fraction * 100}% of data...")
            model_size_hint = pipeline_size_hint.fit(sampled_df)
            
            print(f"Applying VectorSizeHint for column '{col_name}' on full dataset...")
            return model_size_hint.transform(data)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")

    return data

# Applying the function to each vector column
for col_name in vector_cols:
    print(f"Applying VectorSizeHint to vector column: '{col_name}'")
    train_data = apply_vector_size_hint(train_data, col_name)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")
pipeline_final_assembler = Pipeline(stages=[final_assembler])
model_final_assembler = pipeline_final_assembler.fit(train_data)
print("Pipeline Processing Done")

# Step 6: Transform train, valid, and test datasets to include the final features
train_data = model_final_assembler.transform(train_data)

print("Pipeline Transformation Done")

▸,:,


Applying VectorSizeHint to vector column: 'race_vector'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Column 'race_vector' vector size: 1
Fitting VectorSizeHint for column 'race_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'race_vector' on full dataset...


...

Column 'gender_vector' vector size: 1
Fitting VectorSizeHint for column 'gender_vector' on sample of 0.01% of data...
Applying VectorSizeHint for column 'gender_vector' on full dataset...


<IPython.core.display.Javascript object>

Final input columns for feature assembly: ['race_vector', 'gender_vector', 'age_of_TBI_diagnosis', 'MedicalHistory', 'E72', 'Q02', 'S37', 'R52', 'L74', 'T50', 'S87', 'S41', 'C25', 'V58', 'B37', 'T58', 'V41', 'X34', 'C86', 'I01', 'Q11', 'M75', 'B70', 'Q52', 'W27', 'R36', 'V65', 'V28', 'I47', 'K87', 'I52', 'E86', 'Z33', 'Q23', 'Q86', 'N46', 'L62', 'V97', 'T30', 'L50', 'S56', 'H62', 'C91', 'G70', 'Q45', 'T27', 'Z77', 'S51', 'P05', 'F29', 'A83', 'F20', 'W89', 'B68', 'O72', 'X77', 'L87', 'X39', 'B42', 'R59', 'V38', 'X04', 'J35', 'V94', 'W99', 'C54', 'N88', 'F02', 'G63', 'K06', 'J22', 'D78', 'K60', 'W18', 'D21', 'H16', 'L54', 'F11', 'D04', 'H30', 'Z20', 'V73', 'G06', 'K80', 'M19', 'T07', 'W52', 'M80', 'X82', 'H65', 'Z86', 'Y75', 'E55', 'G81', 'V63', 'E87', 'D36', 'O43', 'Z51', 'L91', 'V48', 'K22', 'E73', 'D80', 'T38', 'T84', 'H27', 'Q07', 'E11', 'B00', 'G23', 'E96', 'F33', 'N05', 'Z78', 'D06', 'D74', 'I51', 'B18', 'B58', 'I69', 'R87', 'W50', 'R53', 'C4A', 'O91', 'P02', 'P37', 'M22', 'S81', '

<IPython.core.display.Javascript object>

Pipeline Transformation Done


In [5]:
valid_data = model_final_assembler.transform(valid_data)
test_data = model_final_assembler.transform(test_data)

In [6]:
train_data = train_data.withColumn('label',train_data.label.cast('double'))

In [7]:
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

In [8]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

Number of 0's in the label column: 706360
Number of 1's in the label column: 106714


In [9]:
from pyspark.sql.functions import col

# Number of 0's and 1's in the label column
# count_zeros = 705565
# count_ones = 106624
count_zeros = 706360
count_ones = 106714

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class to match the desired count
upsample_ratio = int((count_zeros - count_ones) / count_ones)
remaining_minority_samples = (count_zeros - count_ones) % count_ones
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(upsample_ratio):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Add remaining samples to reach the exact count
upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df.sample(withReplacement=True, fraction=(remaining_minority_samples / count_ones)))

# Downsampling
# Calculate the fraction for downsampling the majority class
downsample_fraction = count_ones / count_zeros
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

# Combine the upsampled minority class with the downsampled majority class
train_data_balanced= upsampled_minority_class_df.union(downsampled_majority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame:  106921
Number of 1's in the balanced DataFrame:  706578


In [10]:
train_data_balanced.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train3-balanced')

In [15]:
train_data_balanced.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train2-balanced')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
train_data_balanced = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train2-balanced')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
train_data_balanced = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/superset-train3-balanced')

In [12]:
from pyspark.ml.classification import GBTClassifier
gbt = GBTClassifier(featuresCol="features", labelCol='label',maxDepth=10, maxIter=10)
model = gbt.fit(train_data_balanced)

In [18]:
model.save('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/priya-GBT')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
model.save('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/SupersetData/priya-GBT1')

In [19]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

▸,:,


In [14]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

In [15]:
# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
print(f"Train Accuracy: {train_accuracy}")

# Calculate AUC for train data
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")
print(f"Train AUC: {train_auc}")

Train Accuracy: 0.8973
Train AUC: 0.8897


In [16]:
# Step 9: Use the best model (already trained) to transform and evaluate on validation and test data
# Transform validation data
val_data_pred = model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
print(f"Validation Accuracy: {val_accuracy}")

# Calculate AUC for validation data
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
print(f"Validation AUC: {val_auc}")

# Transform test data
test_data_pred = model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
print(f"Test Accuracy: {test_accuracy}")

# Calculate AUC for test data
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
print(f"Test AUC: {test_auc}")

Validation Accuracy: 0.4478
Validation AUC: 0.8707
Test Accuracy: 0.4499
Test AUC: 0.8735


In [17]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [18]:
# # Function to calculate and print confusion matrix metrics
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 1
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        print("Confusion Matrix Metrics:")
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

Confusion Matrix Metrics:
True Positives: 14778
True Negatives: 37367
False Positives: 63357
False Negatives: 407
Accuracy: 0.4499
Precision 1: 0.1891
Recall 1: 0.9732
F1 Score 1: 0.3167
Precision 0: 0.9892
Recall 0: 0.3710
F1 Score 0: 0.5396
Specificity: 0.3710


In [22]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def calculate_confusion_matrix_metrics(predictions: DataFrame, label_col: str, pred_col: str):
    """
    Compute confusion matrix metrics including precision, recall, F1-score, etc.
    """
    tp = predictions.filter((F.col(label_col) == 1) & (F.col(pred_col) == 1)).count()
    tn = predictions.filter((F.col(label_col) == 0) & (F.col(pred_col) == 0)).count()
    fp = predictions.filter((F.col(label_col) == 0) & (F.col(pred_col) == 1)).count()
    fn = predictions.filter((F.col(label_col) == 1) & (F.col(pred_col) == 0)).count()

    # Compute metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # Sensitivity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1_score": f1_score,
        "metrics": {
            "true_positive": tp,
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn
        }
    }

def evaluate_threshold(predictions: DataFrame, label_col: str, probability_col: str, threshold: float):
    """
    Evaluate performance metrics at a given threshold.
    """
    # Assign predictions based on threshold
    predictions = predictions.withColumn(
        "pred_label", (F.col(probability_col)[1] >= threshold).cast("double")
    )
    
    # Compute metrics
    metrics = calculate_confusion_matrix_metrics(predictions, label_col, "pred_label")

    # Print formatted results
    print("\nEvaluation Metrics on Test Data:")
    for metric_name, metric_value in metrics.items():
        if metric_name == "metrics":
            print("\nConfusion Matrix Metrics:")
            for key, value in metric_value.items():
                print(f"{key.replace('_', ' ').title()}: {value}")
        else:
            print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")
    
    return threshold, round(metrics["recall"], 4), round(metrics["specificity"], 4), round(metrics["accuracy"], 4)

def find_best_threshold(predictions: DataFrame, label_col: str, probability_col: str, step: float = 0.05):
    best_threshold = 0.0
    best_score = 0.0
    best_metrics = None
    
    # Iterate over possible thresholds
    for threshold in [round(x, 2) for x in frange(0.0, 1.0, step)]:
        t, sens, spec, acc = evaluate_threshold(predictions, label_col, probability_col, threshold)
        score = sens + spec  # Balance between sensitivity and specificity
        if score > best_score:
            best_score = score
            best_threshold = t
            best_metrics = (t, sens, spec, acc)
    
    return best_metrics

def frange(start, stop, step):
    while start < stop:
        yield start
        start += step

In [23]:
# Apply the trained model to the test data
test_data_pred = model.transform(test_data)

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
print("\nModel Performance Metrics:")
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        print("\nConfusion Matrix Metrics:")
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")


Model Performance Metrics:
Accuracy: 0.4499
Precision: 0.1891
Recall: 0.9732
Specificity: 0.3710
F1 Score: 0.3167

Confusion Matrix Metrics:
True Positive: 14778
True Negative: 37367
False Positive: 63357
False Negative: 407


In [ ]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, DoubleType
import numpy as np

# Define UDF to convert vector to array
def vector_to_array_udf(vec):
    return vec.toArray().tolist()  # Converts vector to array and then to list

# Register UDF
vector_to_array = udf(vector_to_array_udf, ArrayType(DoubleType()))

# Assuming `test_data_pred` is your predictions DataFrame with a 'probability' vector column
test_data_pred = test_data_pred.withColumn("probability_array", vector_to_array(col("probability")))

# Extract the probability of the positive class (e.g., index 1 for binary classification)
test_data_pred = test_data_pred.withColumn("positive_probability", col("probability_array")[1])

# Show the extracted probabilities
test_data_pred.select("positive_probability").show()

# Now, use the `find_best_threshold` function to find the optimal threshold based on sensitivity + specificity
best_threshold, best_sensitivity, best_specificity, best_accuracy = find_best_threshold(
    test_data_pred, "label", "probability_array", step=0.05
)

# Print the best threshold metrics
print("\nBest Threshold Metrics:")
print(f"Best Threshold: {best_threshold:.2f}")
print(f"Sensitivity (Recall): {best_sensitivity:.4f}")
print(f"Specificity: {best_specificity:.4f}")
print(f"Accuracy: {best_accuracy:.4f}")

+--------------------+
|positive_probability|
+--------------------+
|  0.5260280473211039|
| 0.49855705460661426|
| 0.40559181016773715|
| 0.46150592565498383|
|   0.604760782384647|
|  0.5429058349430366|
|  0.9094265868349739|
|  0.7080711289408901|
|  0.8995940519121265|
| 0.40559181016773715|
|  0.5850666774431449|
|  0.8507796914011807|
|  0.5379644748124963|
| 0.46150592565498383|
|  0.7762589252068357|
|  0.7358286322093859|
|  0.7988344531209697|
|  0.5697888206842043|
| 0.49855705460661426|
| 0.40559181016773715|
+--------------------+
only showing top 20 rows


Evaluation Metrics on Test Data:
Accuracy: 0.1310
Precision: 0.1310
Recall: 1.0000
Specificity: 0.0000
F1 Score: 0.2317

Confusion Matrix Metrics:
True Positive: 15185
True Negative: 0
False Positive: 100724
False Negative: 0

Evaluation Metrics on Test Data:
Accuracy: 0.1310
Precision: 0.1310
Recall: 1.0000
Specificity: 0.0000
F1 Score: 0.2317

Confusion Matrix Metrics:
True Positive: 15185
True Negative: 3
False Pos

In [21]:
# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
print(f"Train Accuracy: {train_accuracy}")

# Calculate AUC for train data
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")
print(f"Train AUC: {train_auc}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Train AUC: 0.8872


<IPython.core.display.Javascript object>

In [20]:
# Step 9: Use the best model (already trained) to transform and evaluate on validation and test data
# Transform validation data
val_data_pred = model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
print(f"Validation Accuracy: {val_accuracy}")

# Calculate AUC for validation data
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
print(f"Validation AUC: {val_auc}")

# Transform test data
test_data_pred = model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
print(f"Test Accuracy: {test_accuracy}")

# Calculate AUC for test data
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
print(f"Test AUC: {test_auc}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Test AUC: 0.8713


<IPython.core.display.Javascript object>

In [22]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

▸,:,


In [23]:
# # Function to calculate and print confusion matrix metrics
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 1
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        print("Confusion Matrix Metrics:")
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Confusion Matrix Metrics:
True Positives: 14944
True Negatives: 24173
False Positives: 76569
False Negatives: 324
Accuracy: 0.3372
Precision 1: 0.1633
Recall 1: 0.9788
F1 Score 1: 0.2799
Precision 0: 0.9868
Recall 0: 0.2399
F1 Score 0: 0.3860
Specificity: 0.2399


<IPython.core.display.Javascript object>